# GSM8K：固定預算 LoRA 實驗

比較基礎模型、隨機選樣 LoRA 與中等損失選樣 LoRA。前半部可直接在 CPU 上執行，使用 GitHub 內的 RTX 5090 或 RTX 5080 精簡結果；後半部提供完整產物核對、逐題分析及 GPU 實驗。

建議安裝：`python -m pip install -e ".[notebook]"`。GPU 區段另需 `.[train]` 與 CUDA 版 PyTorch。


In [ ]:
from pathlib import Path
import csv
import json
import sys
from decimal import Decimal

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file()), None)
if ROOT is None:
    raise RuntimeError("請從倉庫根目錄或 notebooks 目錄啟動 kernel")
sys.path.insert(0, str(ROOT / "src"))
from gsm8k_lora.common import extract_answer, load_config, read_json, read_jsonl

print("Repository:", ROOT)
print("Python:", sys.version.split()[0])


## 1. 實驗參數

可切換 validation / test 比較圖表。GPU 執行與網路下載透過下方開關啟用；預設執行結果分析。


In [ ]:
EXPERIMENT = "seed13"             # "seed13" (RTX 5090) or "rtx5080" (RTX 5080)
SPLIT = "test"                   # "validation" 或 "test"
FETCH_ARTIFACTS = False           # 從 Hugging Face 還原至空的 artifacts 目錄
RUN_FULL_AUDIT = False            # 重算所有保存的預測、選樣與 bootstrap
RUN_GPU_PREDICT = False           # 載入基礎或 LoRA 模型，回答自訂問題
RUN_GPU_TRAIN = False             # 在獨立目錄跑短訓練
METHOD = "moderate"               # "base", "random", "moderate"
SEED = 13
QUESTION = "A shop has 12 apples and sells 5. How many apples remain?"
EXPERIMENT_NAME = "notebook-trial-01"
DEBUG_STEPS = 10
TRAIN_BATCH_SIZE = 2              # 1, 2, or 4; effective training batch stays 32

assert TRAIN_BATCH_SIZE in {1, 2, 4}
assert SPLIT in {"validation", "test"}
assert METHOD in {"base", "random", "moderate"}
assert EXPERIMENT in {"seed13", "rtx5080"}
SNAPSHOT = ROOT / "results" / EXPERIMENT
ARTIFACTS = ROOT / "artifacts"
if EXPERIMENT == "rtx5080":
    ARTIFACTS = ARTIFACTS / "reproduction-rtx5080"
print("Experiment:", EXPERIMENT)
print("Local files:", ARTIFACTS)


## 2. 主結果與圖表

準確率和格式率在 CSV 中以 0–1 儲存。成本欄位以分鐘儲存，圖表換算為秒。


In [ ]:
from IPython.display import Markdown, display

with (SNAPSHOT / f"summary_{SPLIT}.csv").open(encoding="utf-8", newline="") as handle:
    rows = list(csv.DictReader(handle))
labels = ["Base", "Random", "Moderate"]
table = ["| Method | Accuracy | Format | Mean tokens | Score + train (s) |", "| --- | ---: | ---: | ---: | ---: |"]
for label, row in zip(labels, rows):
    cost = 60 * (float(row["score_minutes"]) + float(row["train_minutes"]))
    table.append(f"| {label} | {100*float(row['accuracy']):.2f}% | {100*float(row['strict_format_rate']):.2f}% | {float(row['mean_generated_tokens']):.2f} | {cost:.2f} |")
display(Markdown("\n".join(table)))


In [ ]:
import matplotlib.pyplot as plt

colors = ["#425b76", "#24968b", "#df9a39"]
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8), layout="constrained")
for ax, key, title, scale in zip(axes,
    ["accuracy", "strict_format_rate", "mean_generated_tokens"],
    ["Answer accuracy (%)", "Answer format (%)", "Mean generated tokens"], [100, 100, 1]):
    values = [scale * float(row[key]) for row in rows]
    bars = ax.bar(labels, values, color=colors)
    ax.bar_label(bars, fmt="%.2f", padding=4)
    ax.set(title=title, ylim=(0, max(values) * 1.2))
    ax.spines[["top", "right"]].set_visible(False)
fig.suptitle(f"GSM8K / {EXPERIMENT} / {SPLIT} / training seed 13")
plt.show()


## 3. 預算匹配與配對差異

T 為完整非 padding 訓練 token，T_y 為參與 loss 的回答 token。Bootstrap 區間描述固定 B/C 模型在題目抽樣下的不確定性。


In [ ]:
random_row, moderate_row = rows[1], rows[2]
for key, tolerance in [("T", 0.01), ("T_y", 0.05), ("unique_samples", 0.05)]:
    first, second = int(random_row[key]), int(moderate_row[key])
    gap = abs(first - second) / max(first, second, 1)
    assert gap <= tolerance, (key, gap)
    print(f"{key}: B={first:,}, C={second:,}, relative gap={gap:.4%}")
comparison = read_json(SNAPSHOT / f"comparison_{SPLIT}.json")
pair = comparison["pairs"][0]
print(f"C - B = {pair['C_minus_B_pp']:+.4f} percentage points")
print("Paired bootstrap 95% CI:", pair["paired_bootstrap_95_ci_pp"])


## 4. 數值答案解析測試

可修改範例，檢查貨幣、千位逗號、負數、缺少標記與無效欄位。評估沿用相同的解析函式。


In [ ]:
cases = [
    ("#### 7", Decimal("7")),
    ("#### $1,250.00", Decimal("1250.00")),
    ("#### −3.5", Decimal("-3.5")),
    ("The answer is 7 apples.", Decimal("7")),
    ("#### invalid", None),
    ("#### 1/2", None),
    ("#### 1e3", None),
]
for text, expected in cases:
    parsed = extract_answer(text)
    assert parsed == expected, (text, parsed, expected)
    print(repr(text), "->", parsed)
assert extract_answer("The answer is 7.", strict=True) is None
print("All parser examples passed.")


## 5. Experiment files and verification

Choose `EXPERIMENT="seed13"` for the RTX 5090 experiment or `"rtx5080"` for the RTX 5080 experiment. Hugging Face preserves prepared data, scores, selections, predictions, adapters, and logs. Downloads require an empty destination. Existing local files can be verified directly.


In [ ]:
if FETCH_ARTIFACTS:
    from gsm8k_lora.artifacts import fetch
    record = read_json(SNAPSHOT / "artifacts.json")
    fetch(record, ARTIFACTS)
if RUN_FULL_AUDIT:
    from gsm8k_lora.reproduce import reproduce
    report = reproduce(ARTIFACTS)
    display(report)
else:
    report = read_json(SNAPSHOT / "reproduction.json")
    print("Published audit:", report["status"], "/", report["verified_predictions"], "predictions")


## 6. 逐題比較

完整產物存在時，找出 C 修正 B 或 B 修正 C 的題目。修改 `CASE_INDEX` 可查看其他案例。


In [ ]:
CASE_INDEX = 0
prediction_paths = [ARTIFACTS / "runs" / name / f"eval_{SPLIT}" / "predictions.jsonl" for name in ["base", "random_13", "moderate_13"]]
if all(path.is_file() for path in prediction_paths):
    predictions = [{row["id"]: row for row in read_jsonl(path)} for path in prediction_paths]
    disagreements = sorted(identifier for identifier in predictions[1] if predictions[1][identifier]["correct"] != predictions[2][identifier]["correct"])
    print("B/C disagreements:", len(disagreements))
    identifier = disagreements[CASE_INDEX % len(disagreements)] if disagreements else None
    if identifier:
        print("ID:", identifier, "\nQuestion:", predictions[0][identifier]["question"])
        print("Gold:", predictions[0][identifier]["gold"])
        for label, records in zip(labels, predictions):
            print(f"\n{label} / correct={records[identifier]['correct']}\n{records[identifier]['output']}")
else:
    print("Download the full experiment directory to enable per-question analysis.")


## 7. GPU 推理

啟用 `RUN_GPU_PREDICT` 後，使用與正式評估相同的生成函式。需要 CUDA BF16 環境、準備資料與指定方法的 adapter。


In [ ]:
if RUN_GPU_PREDICT:
    import gc
    import os
    os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
    from gsm8k_lora.common import load_prepared, require_cuda
    from gsm8k_lora.run import get_inference_model, generate
    config_path = ARTIFACTS / "config.json"
    config = load_config(config_path if config_path.is_file() else ROOT / "configs" / "default.json")
    config.update(data_dir=str(ARTIFACTS / "data"), output_dir=str(ARTIFACTS / "runs"))
    require_cuda()
    data_root, metadata = load_prepared(config)
    model, tokenizer, _ = get_inference_model(config, data_root, metadata, METHOD, SEED)
    answer = generate(model, tokenizer, [QUESTION], config)[0]
    print(answer["output"])
    print("Parsed:", extract_answer(answer["output"]))
    del model, tokenizer
    gc.collect()
    import torch
    torch.cuda.empty_cache()
else:
    print("Set RUN_GPU_PREDICT=True to generate an answer on GPU.")


## 8. 獨立訓練試驗

使用已有種子 13 清單，將短訓練保存到 `artifacts/notebook/<EXPERIMENT_NAME>/debug_*`。可修改學習率、LoRA rank 或 `DEBUG_STEPS`。比較方法時請統一設定。完整實驗入口為 `python scripts/reproduce_gpu.py`。

`TRAIN_BATCH_SIZE` controls GPU memory use; accumulation keeps the effective batch at 32. The output directory follows the selected experiment.


In [ ]:
if RUN_GPU_TRAIN:
    import os
    import subprocess
    os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
    trial = ARTIFACTS / "notebook" / EXPERIMENT_NAME
    trial.mkdir(parents=True, exist_ok=False)
    config_path = ARTIFACTS / "config.json"
    trial_config = load_config(config_path if config_path.is_file() else ROOT / "configs" / "default.json")
    trial_config.update(data_dir=str(ARTIFACTS / "data"), output_dir=str(trial))
    trial_config["train"]["learning_rate"] = 1e-4
    trial_config["train"]["batch_size"] = TRAIN_BATCH_SIZE
    trial_config["train"]["gradient_accumulation_steps"] = 32 // TRAIN_BATCH_SIZE
    trial_path = trial / "config.json"
    trial_path.write_text(json.dumps(trial_config, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    train_method = METHOD if METHOD in {"random", "moderate"} else "random"
    subprocess.run([sys.executable, "-m", "gsm8k_lora.run", "--config", str(trial_path), "train", "--method", train_method, "--seed", str(SEED), "--debug-steps", str(DEBUG_STEPS)], cwd=ROOT, check=True)
    display(read_json(trial / f"debug_{train_method}_{SEED}" / "train.json"))
else:
    print("Set RUN_GPU_TRAIN=True to run an isolated training trial.")


## 延伸閱讀

- [實驗方法](../docs/methodology.md)
- [RTX 5090 結果分析](../docs/experiment-analysis.md)
- [GPU 操作流程](../docs/server-workflow.md)
- [完整實驗產物](https://huggingface.co/datasets/tomtom123/7021final-artifacts)
